# vectorbt 实现 Top1 ETF 轮动

本 notebook 调用项目脚本 `scripts/vectorbt_top1_etf_rotation.py`，用 vectorbt 执行 Top1 ETF 轮动回测。

规则：

- 行业/主题 ETF 中按过去 `42` 个交易日动量排序；
- 每月末生成信号，下一交易日目标权重生效；
- 若 Top1 行业 ETF 动量为正，满仓持有；
- 若 Top1 行业 ETF 动量为负或不可用，切到正动量防守资产；
- 若防守资产也无正动量，则空仓；
- vectorbt 成本：佣金 `0.01%`，滑点 `0.01%`。

In [ ]:
# ? pathlib ?? Path?????????????
from pathlib import Path
# ?? pandas???????? CSV ?????
import pandas as pd

# ?????? notebook ?????????
BASE = Path.cwd()
# ????????? notebooks ??????????????
if BASE.name == "notebooks":
    # ??????? notebooks ????????????????
    BASE = BASE.parent

# ?? Top1 ETF ????????????
SCRIPT_PATH = BASE / "scripts" / "vectorbt_top1_etf_rotation.py"
# ? notebook ?????????????? %run ?????????
SCRIPT_PATH

## 1. 执行回测脚本

In [ ]:
# ? notebook ???????????????????????????????
# %run ?? SCRIPT_PATH ??? Python ??????????
%run $SCRIPT_PATH

## 2. 读取指标与最新信号

In [ ]:
# ?????? CSV ?????
metrics_path = BASE / "outputs" / "etf_top1_rotation_vectorbt_metrics.csv"
# ???????? CSV ?????
decisions_path = BASE / "outputs" / "etf_top1_rotation_vectorbt_decisions.csv"
# ?? vectorbt ???? CSV ?????
orders_path = BASE / "outputs" / "etf_top1_rotation_vectorbt_orders.csv"
# ??????????? CSV ?????
daily_path = BASE / "outputs" / "etf_top1_rotation_vectorbt_daily_value.csv"

# ?????????????????????????
metrics = pd.read_csv(metrics_path)
# ?????????? signal_date ????????
decisions = pd.read_csv(decisions_path, parse_dates=["signal_date"])
# ?? vectorbt ?????????
orders = pd.read_csv(orders_path)
# ?????????? date ????????
daily = pd.read_csv(daily_path, parse_dates=["date"])

# ?????????????????????
metrics

In [ ]:
# ???? 10 ????????????????????
decisions.tail(10)

## 3. 查看 vectorbt 订单

In [ ]:
# ??? 20 ??????????????????????
orders.head(20)

In [ ]:
# ???? 20 ???????????????????
orders.tail(20)

## 4. 查看净值曲线数据

In [ ]:
# ????????????????????????
daily.tail()

## 5. 收益率图

In [ ]:
# ?? matplotlib ??????
import matplotlib.pyplot as plt
# ??????????????????
import matplotlib.font_manager as fm
# ????????????????????????
from matplotlib.ticker import PercentFormatter

# ??????? Windows ???????????
font_candidates = [
    # ?????????
    Path("C:/Windows/Fonts/msyh.ttc"),
    # ???????
    Path("C:/Windows/Fonts/simhei.ttf"),
    # ???????
    Path("C:/Windows/Fonts/simsun.ttc"),
    # ???????
    Path("C:/Windows/Fonts/simkai.ttf"),
]
# ?????????????????????????????? None?
font_path = next((path for path in font_candidates if path.exists()), None)
# ??????????????????? matplotlib ?????
if font_path is not None:
    # ??????? matplotlib ???????
    fm.fontManager.addfont(str(font_path))
    # ???????????????
    font_prop = fm.FontProperties(fname=str(font_path))
    # ????????????????????
    plt.rcParams["font.family"] = font_prop.get_name()
# ????????????????????????????
plt.rcParams["axes.unicode_minus"] = False

# ???????????????????????????? daily ??
plot_daily = daily.sort_values("date").copy()
# ????? 1???????????????????
plot_daily["cumulative_return"] = plot_daily["nav"] - 1.0

# ?????????????????????????????????
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

# ????????????????
axes[0].plot(plot_daily["date"], plot_daily["cumulative_return"], color="#2563eb", linewidth=1.8)
# ??????????? 0% ????
axes[0].axhline(0, color="#64748b", linewidth=0.8, linestyle="--")
# ???????????
axes[0].set_title("Top1 ETF ???????")
# ?????????????
axes[0].set_ylabel("?????")
# ??????????????????????
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
# ???????????????????
axes[0].grid(True, alpha=0.25)

# ????????????????
axes[1].plot(plot_daily["date"], plot_daily["daily_return"], color="#0f766e", linewidth=0.8, alpha=0.85)
# ?????????? 0% ????
axes[1].axhline(0, color="#64748b", linewidth=0.8, linestyle="--")
# ??????????
axes[1].set_title("????")
# ????????????
axes[1].set_ylabel("????")
# ?????????????????????
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
# ??????????????????
axes[1].grid(True, alpha=0.25)

# ?????????????????????
fig.autofmt_xdate()
# ?????????????????????????
fig.tight_layout()

# ???????????
return_chart_path = BASE / "outputs" / "etf_top1_rotation_vectorbt_return_chart.png"
# ?????? PNG ???????? dpi ??????
fig.savefig(return_chart_path, dpi=160, bbox_inches="tight")
# ???????????? notebook ?????????
return_chart_path

## 6. 输出文件

- 每日净值：`outputs/etf_top1_rotation_vectorbt_daily_value.csv`
- 收益率图：`outputs/etf_top1_rotation_vectorbt_return_chart.png`
- 目标权重：`outputs/etf_top1_rotation_vectorbt_target_weights.csv`
- 调仓决策：`outputs/etf_top1_rotation_vectorbt_decisions.csv`
- vectorbt 订单：`outputs/etf_top1_rotation_vectorbt_orders.csv`
- 回测指标：`outputs/etf_top1_rotation_vectorbt_metrics.csv`
- Markdown 报告：`outputs/etf_top1_rotation_vectorbt_report.md`